In [12]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import binomtest


# ============================================================
# Reading binary experiment data
# ============================================================

def read_experiment_data(csv_path, p=None):
    """
    Expected format:
      - rows = vertices / atoms
      - columns = observations / shots

    Returns:
      X with shape (n_observations, p).
    """
    df = pd.read_csv(csv_path, header=None)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    X_raw = df.to_numpy()
    values = np.unique(X_raw)

    return X_raw.T.astype(int)


# ============================================================
# Graph support utilities
# ============================================================

def valid_rows_mask(X, edges):
    """Rows satisfying x_u x_v = 0 for every edge (u, v)."""
    ok = np.ones(X.shape[0], dtype=bool)
    for u, v in edges:
        ok &= ~((X[:, u] == 1) & (X[:, v] == 1))
    return ok


def enumerate_independent_set_masks(p, edges):
    masks = []

    for mask in range(1 << p):
        good = True

        for u, v in edges:
            if ((mask >> u) & 1) and ((mask >> v) & 1):
                good = False
                break

        if good:
            masks.append(mask)

    return np.array(masks, dtype=np.int64)


def masks_to_matrix(masks, p):
    masks = np.asarray(masks, dtype=np.int64)
    return ((masks[:, None] >> np.arange(p, dtype=np.int64)) & 1).astype(int)


def matrix_to_masks(X):
    p = X.shape[1]
    powers = 1 << np.arange(p, dtype=np.int64)
    return (X.astype(np.int64) * powers).sum(axis=1)


def mask_to_bitstring(mask, p):
    return "".join(str((mask >> j) & 1) for j in range(p))


def mask_to_profile(mask, p):
    active = [f"v{j + 1}" for j in range(p) if (mask >> j) & 1]
    return "none" if len(active) == 0 else " + ".join(active)


# ============================================================
# Fitting mult_G(1, y)
# ============================================================

def fit_multG1_from_state_counts(
    state_counts,
    support_states,
    theta_init=None,
    theta_bounds=(-20.0, 20.0),
):
    counts = np.asarray(state_counts, dtype=float)
    S = np.asarray(support_states, dtype=float)

    n = counts.sum()
    sufficient_stat = counts @ S
    p = S.shape[1]

    if theta_init is None:
        theta_init = np.zeros(p, dtype=float)
    else:
        theta_init = np.asarray(theta_init, dtype=float)

    def objective(theta):
        eta = S @ theta
        logZ = logsumexp(eta)
        probs = np.exp(eta - logZ)

        nll = n * logZ - sufficient_stat @ theta
        grad = n * (probs @ S) - sufficient_stat
        return nll, grad

    lo, hi = theta_bounds
    bounds = [(lo, hi)] * p

    opt = minimize(
        fun=lambda th: objective(th)[0],
        x0=theta_init,
        jac=lambda th: objective(th)[1],
        method="L-BFGS-B",
        bounds=bounds,
    )

    theta_hat = opt.x
    eta_hat = S @ theta_hat
    logZ_hat = logsumexp(eta_hat)
    probs_hat = np.exp(eta_hat - logZ_hat)

    return {
        "theta_hat": theta_hat,
        "y_hat": np.exp(theta_hat),
        "logZ_hat": float(logZ_hat),
        "loglik_hat": float(sufficient_stat @ theta_hat - n * logZ_hat),
        "probs_hat": probs_hat,
        "expected_state_counts": n * probs_hat,
        "fitted_marginals": probs_hat @ S,
        "optimizer_success": bool(opt.success),
        "optimizer_message": str(opt.message),
    }


def deviance_statistic(counts, probs):
    counts = np.asarray(counts, dtype=float)
    probs = np.asarray(probs, dtype=float)

    n = counts.sum()
    expected = n * probs

    positive = counts > 0
    return float(2.0 * np.sum(counts[positive] * np.log(counts[positive] / expected[positive])))


# ============================================================
# Main support-restricted GOF test
# ============================================================

def test_filtered_multG1_from_csv(
    csv_path,
    edges,
    graph_name,
    p=4,
    bootstrap_B=300,
    refit_in_bootstrap=True,
    seed=123,
):
    X = read_experiment_data(csv_path, p=p)
    n_total = X.shape[0]

    # Restrict observations to the graph support.
    is_valid = valid_rows_mask(X, edges)
    X_valid = X[is_valid]
    n_valid = X_valid.shape[0]
    n_invalid = n_total - n_valid

    # Enumerate exact independent-set support.
    support_masks = enumerate_independent_set_masks(p, edges)
    support_states = masks_to_matrix(support_masks, p)
    mask_to_index = {int(m): i for i, m in enumerate(support_masks.tolist())}

    # Count observed states on the support.
    valid_masks = matrix_to_masks(X_valid)
    obs_counts = np.zeros(len(support_masks), dtype=int)

    for m in valid_masks:
        obs_counts[mask_to_index[int(m)]] += 1

    # Fit mult_G(1, y).
    fit = fit_multG1_from_state_counts(
        state_counts=obs_counts,
        support_states=support_states,
    )

    D_obs = deviance_statistic(obs_counts, fit["probs_hat"])

    # Parametric bootstrap, with refitting by default.
    rng = np.random.default_rng(seed)
    D_boot = np.empty(bootstrap_B, dtype=float)

    for b in range(bootstrap_B):
        boot_counts = rng.multinomial(n_valid, fit["probs_hat"])

        if refit_in_bootstrap:
            fit_b = fit_multG1_from_state_counts(
                state_counts=boot_counts,
                support_states=support_states,
                theta_init=fit["theta_hat"],
            )
            D_boot[b] = deviance_statistic(boot_counts, fit_b["probs_hat"])
        else:
            D_boot[b] = deviance_statistic(boot_counts, fit["probs_hat"])

    p_value = (1.0 + np.sum(D_boot >= D_obs)) / (bootstrap_B + 1.0)

    state_table = pd.DataFrame({
        "state_mask": support_masks,
        "state": [mask_to_bitstring(int(m), p) for m in support_masks],
        "profile": [mask_to_profile(int(m), p) for m in support_masks],
        "obs": obs_counts,
        "exp": fit["expected_state_counts"],
        "prob_hat": fit["probs_hat"],
    })

    state_table["signed_pearson_resid"] = (
        (state_table["obs"] - state_table["exp"]) / np.sqrt(state_table["exp"])
    )
    state_table["abs_pearson_resid"] = np.abs(state_table["signed_pearson_resid"])

    state_table = state_table.sort_values(
        ["abs_pearson_resid", "obs"],
        ascending=[False, False],
    ).reset_index(drop=True)

    marginal_table = pd.DataFrame({
        "vertex": [f"v{j + 1}" for j in range(p)],
        "obs_mean_filtered": X_valid.mean(axis=0),
        "fit_mean": fit["fitted_marginals"],
        "diff": X_valid.mean(axis=0) - fit["fitted_marginals"],
        "y_hat": fit["y_hat"],
        "theta_hat": fit["theta_hat"],
    })

    return {
        "summary": {
            "csv_path": csv_path,
            "graph": graph_name,
            "edges": edges,
            "n_total": int(n_total),
            "n_valid": int(n_valid),
            "n_invalid": int(n_invalid),
            "frac_valid": float(n_valid / n_total),
            "support_size": int(len(support_masks)),
            "deviance_obs": float(D_obs),
            "bootstrap_B": int(bootstrap_B),
            "bootstrap_p_value": float(p_value),
            "refit_in_bootstrap": bool(refit_in_bootstrap),
            "reject_multG_at_05": bool(p_value < 0.05),
            "optimizer_success": bool(fit["optimizer_success"]),
            "optimizer_message": fit["optimizer_message"],
        },
        "fit": fit,
        "state_table": state_table,
        "marginal_table": marginal_table,
        "X_valid": X_valid,
        "valid_rows_mask": is_valid,
        "support_states": support_states,
        "support_masks": support_masks,
        "obs_counts": obs_counts,
        "D_boot": D_boot,
    }


# ============================================================
# Reporting
# ============================================================

def print_report_filtered_multG1(result, top_k=20, digits=4):
    s = result["summary"]

    print("=== SUPPORT-RESTRICTED multG(1,y) GOF ===")
    print(f"csv_path             = {s['csv_path']}")
    print(f"graph                = {s['graph']}")
    print(f"edges                = {s['edges']}")
    print(f"n_total              = {s['n_total']}")
    print(f"n_valid              = {s['n_valid']}")
    print(f"n_invalid            = {s['n_invalid']}")
    print(f"frac_valid           = {s['frac_valid']:.{digits}f}")
    print(f"support_size         = {s['support_size']}")
    print(f"deviance_obs         = {s['deviance_obs']:.{digits}f}")
    print(f"bootstrap_B          = {s['bootstrap_B']}")
    print(f"bootstrap_p_value    = {s['bootstrap_p_value']:.{digits}f}")
    print(f"refit_in_bootstrap   = {s['refit_in_bootstrap']}")
    print(f"reject_multG_at_05   = {s['reject_multG_at_05']}")
    print(f"optimizer_success    = {s['optimizer_success']}")
    print(f"optimizer_message    = {s['optimizer_message']}")
    print()

    print("=== MARGINALS ON ADMISSIBLE OBSERVATIONS ===")
    print(result["marginal_table"].round(digits).to_string(index=False))
    print()

    print(f"=== TOP {top_k} STATES BY ABSOLUTE PEARSON RESIDUAL ===")
    print(result["state_table"].head(top_k).round(digits).to_string(index=False))
    print()

In [13]:
# ============================================================
# Settings
# ============================================================

CSV_PATH = "experiment_data_fig2d.csv"
GRAPH_NAME = "P4 path: v1-v2-v3-v4"

# 0 = v1, 1 = v2, 2 = v3, 3 = v4
EDGES = [(0, 1), (1, 2), (2, 3)]

P = 4
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123

# ============================================================
# Run support-restricted GOF
# ============================================================

result = test_filtered_multG1_from_csv(
    csv_path=CSV_PATH,
    edges=EDGES,
    graph_name=GRAPH_NAME,
    p=P,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_filtered_multG1(result, top_k=20, digits=4)


=== SUPPORT-RESTRICTED multG(1,y) GOF ===
csv_path             = experiment_data_fig2d.csv
graph                = P4 path: v1-v2-v3-v4
edges                = [(0, 1), (1, 2), (2, 3)]
n_total              = 678
n_valid              = 191
n_invalid            = 487
frac_valid           = 0.2817
support_size         = 8
deviance_obs         = 0.5081
bootstrap_B          = 500
bootstrap_p_value    = 0.9102
refit_in_bootstrap   = True
reject_multG_at_05   = False
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS ON ADMISSIBLE OBSERVATIONS ===
vertex  obs_mean_filtered  fit_mean  diff   y_hat  theta_hat
    v1             0.4398    0.4398  -0.0  3.1111     1.1350
    v2             0.4188    0.4188   0.0 11.3410     2.4284
    v3             0.4293    0.4293  -0.0 14.0093     2.6397
    v4             0.4555    0.4555   0.0  3.9545     1.3749

=== TOP 20 STATES BY ABSOLUTE PEARSON RESIDUAL ===
 state_mask state profile  obs

In [14]:
# ============================================================
# Settings
# ============================================================

CSV_PATH = "experiment_data_fig2j.csv"
GRAPH_NAME = "Star on 4 vertices: v3 connected to v1, v2, v4"

# 0 = v1, 1 = v2, 2 = v3, 3 = v4
EDGES = [(0, 2), (1, 2), (2, 3)]

P = 4
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123

# ============================================================
# Run support-restricted GOF
# ============================================================

result = test_filtered_multG1_from_csv(
    csv_path=CSV_PATH,
    edges=EDGES,
    graph_name=GRAPH_NAME,
    p=P,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_filtered_multG1(result, top_k=20, digits=4)


=== SUPPORT-RESTRICTED multG(1,y) GOF ===
csv_path             = experiment_data_fig2j.csv
graph                = Star on 4 vertices: v3 connected to v1, v2, v4
edges                = [(0, 2), (1, 2), (2, 3)]
n_total              = 425
n_valid              = 183
n_invalid            = 242
frac_valid           = 0.4306
support_size         = 9
deviance_obs         = 8.9469
bootstrap_B          = 500
bootstrap_p_value    = 0.0918
refit_in_bootstrap   = True
reject_multG_at_05   = False
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS ON ADMISSIBLE OBSERVATIONS ===
vertex  obs_mean_filtered  fit_mean  diff   y_hat  theta_hat
    v1             0.2022    0.2022  -0.0  1.7619     0.5664
    v2             0.2459    0.2459  -0.0  3.4615     1.2417
    v3             0.6831    0.6831   0.0 77.0145     4.3440
    v4             0.2077    0.2077   0.0  1.9000     0.6419

=== TOP 20 STATES BY ABSOLUTE PEARSON RESIDUAL ===
 sta

In [15]:
# ============================================================
# Settings
# ============================================================

CSV_PATH = "experiment_data_fig2e.csv"
GRAPH_NAME = "C4 cycle: v1-v2-v3-v4-v1"

# 0 = v1, 1 = v2, 2 = v3, 3 = v4
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0)]

P = 4
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123

# ============================================================
# Run support-restricted GOF
# ============================================================

result = test_filtered_multG1_from_csv(
    csv_path=CSV_PATH,
    edges=EDGES,
    graph_name=GRAPH_NAME,
    p=P,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_filtered_multG1(result, top_k=20, digits=4)


=== SUPPORT-RESTRICTED multG(1,y) GOF ===
csv_path             = experiment_data_fig2e.csv
graph                = C4 cycle: v1-v2-v3-v4-v1
edges                = [(0, 1), (1, 2), (2, 3), (3, 0)]
n_total              = 672
n_valid              = 261
n_invalid            = 411
frac_valid           = 0.3884
support_size         = 7
deviance_obs         = 2.7639
bootstrap_B          = 500
bootstrap_p_value    = 0.2275
refit_in_bootstrap   = True
reject_multG_at_05   = False
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS ON ADMISSIBLE OBSERVATIONS ===
vertex  obs_mean_filtered  fit_mean  diff   y_hat  theta_hat
    v1             0.4904    0.4904  -0.0 36.8408     3.6066
    v2             0.4674    0.4674   0.0 15.7429     2.7564
    v3             0.4713    0.4713  -0.0 14.5144     2.6751
    v4             0.4828    0.4828   0.0 33.6042     3.5146

=== TOP 20 STATES BY ABSOLUTE PEARSON RESIDUAL ===
 state_mask state 

In [16]:
# ============================================================
# Settings
# ============================================================

CSV_PATH = "experiment_data_fig2k.csv"
GRAPH_NAME = "Paw graph: triangle v1-v2-v3-v1 plus edge v3-v4"

# 0 = v1, 1 = v2, 2 = v3, 3 = v4
EDGES = [(0, 1), (1, 2), (2, 0), (2, 3)]

P = 4
BOOTSTRAP_B = 500
REFIT_IN_BOOTSTRAP = True
SEED = 123

# ============================================================
# Run support-restricted GOF
# ============================================================

result = test_filtered_multG1_from_csv(
    csv_path=CSV_PATH,
    edges=EDGES,
    graph_name=GRAPH_NAME,
    p=P,
    bootstrap_B=BOOTSTRAP_B,
    refit_in_bootstrap=REFIT_IN_BOOTSTRAP,
    seed=SEED,
)

print_report_filtered_multG1(result, top_k=20, digits=4)


=== SUPPORT-RESTRICTED multG(1,y) GOF ===
csv_path             = experiment_data_fig2k.csv
graph                = Paw graph: triangle v1-v2-v3-v1 plus edge v3-v4
edges                = [(0, 1), (1, 2), (2, 0), (2, 3)]
n_total              = 525
n_valid              = 73
n_invalid            = 452
frac_valid           = 0.1390
support_size         = 7
deviance_obs         = 1.5922
bootstrap_B          = 500
bootstrap_p_value    = 0.5289
refit_in_bootstrap   = True
reject_multG_at_05   = False
optimizer_success    = True
optimizer_message    = CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== MARGINALS ON ADMISSIBLE OBSERVATIONS ===
vertex  obs_mean_filtered  fit_mean  diff   y_hat  theta_hat
    v1             0.2466    0.2466  -0.0  9.0000     2.1972
    v2             0.1918    0.1918  -0.0  7.0000     1.9459
    v3             0.5342    0.5342   0.0 47.3568     3.8577
    v4             0.2740    0.2740   0.0  1.4286     0.3567

=== TOP 20 STATES BY ABSOLUTE PEARSON RESIDUAL 